### Import libraries

In [31]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings 
warnings.filterwarnings("ignore")

### Load Dataset

In [32]:
df = pd.read_csv("../datasets/resume_dataset_1200.csv")
df.head()

,Name,Age,Gender,Education_Level,Field_of_Study,Degrees,Institute_Name,Graduation_Year,Experience_Years,Current_Job_Title,Previous_Job_Titles,Skills,Certifications,Target_Job_Description
0,Akash Pillai,30,Non-Binary,Master's,Cybersecurity,Master's in Cybersecurity,University of Pennsylvania,2025,0,NaN,NaN,"Node.js, JavaScript, Deep Learning, Statistics...",Google Cloud Professional,Seeking a challenging role as a Software Devel...
1,Charlotte Taylor,27,Non-Binary,Bachelor's,Electronics Engineering,Bachelor's in Electronics Engineering,Pune University,2019,5,Cybersecurity Engineer,NaN,"Spark, Kubernetes, Terraform, Natural Language...",TensorFlow Developer Certificate,Targeting a Cybersecurity Engineer position to...
2,James Zhou,45,Male,Bachelor's,Computer Science,Bachelor's in Computer Science,Amity University,2023,2,Prompt Engineer,NaN,"Data Analysis, Node.js, Machine Learning, Linu...","Microsoft Azure Fundamentals, Cisco Certified ...",Targeting a Prompt Engineer position to utiliz...
3,Amelia Thomas,28,Male,Master's,Information Technology,Master's in Information Technology,University of Pennsylvania,2023,2,AI Engineer,NaN,"Docker, Kubernetes, Blockchain, Spark",NaN,Targeting a Data Scientist position where I ca...
4,Amanda Jain,42,Non-Binary,Master's,Electronics Engineering,Master's in Electronics Engineering,University of Toronto,2023,2,Cybersecurity Engineer,NaN,"Statistics, Blockchain, Kubernetes, Cybersecurity",NaN,Looking for an opportunity as a Prompt Enginee...


### Check Columns Names

In [33]:
df.columns

Index(['Name', 'Age', 'Gender', 'Education_Level', 'Field_of_Study', 'Degrees',
       'Institute_Name', 'Graduation_Year', 'Experience_Years',
       'Current_Job_Title', 'Previous_Job_Titles', 'Skills', 'Certifications',
       'Target_Job_Description'],
      dtype='object')

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Name                    1200 non-null   object
 1   Age                     1200 non-null   int64 
 2   Gender                  1200 non-null   object
 3   Education_Level         1200 non-null   object
 4   Field_of_Study          1200 non-null   object
 5   Degrees                 1200 non-null   object
 6   Institute_Name          1200 non-null   object
 7   Graduation_Year         1200 non-null   int64 
 8   Experience_Years        1200 non-null   int64 
 9   Current_Job_Title       759 non-null    object
 10  Previous_Job_Titles     286 non-null    object
 11  Skills                  1200 non-null   object
 12  Certifications          881 non-null    object
 13  Target_Job_Description  1200 non-null   object
dtypes: int64(3), object(11)
memory usage: 131.4+ KB


In [35]:
df.isnull().sum()

Name                        0
Age                         0
Gender                      0
Education_Level             0
Field_of_Study              0
Degrees                     0
Institute_Name              0
Graduation_Year             0
Experience_Years            0
Current_Job_Title         441
Previous_Job_Titles       914
Skills                      0
Certifications            319
Target_Job_Description      0
dtype: int64

In [36]:
df["resume_text"] = (
    df["Field_of_Study"].fillna("") + " " +
    df["Degrees"].fillna("") + " " +
    df["Experience_Years"].fillna(0).astype(str) + " years experience" +
    df["Current_Job_Title"].fillna("") + " " +
    df["Previous_Job_Titles"].fillna("") + " "+
    df["Skills"].fillna("") + " " +
    df["Certifications"].fillna("")
)

In [37]:
df["resume_text"]

0       Cybersecurity Master's in Cybersecurity 0 year...
1       Electronics Engineering Bachelor's in Electron...
2       Computer Science Bachelor's in Computer Scienc...
3       Information Technology Master's in Information...
4       Electronics Engineering Master's in Electronic...
                              ...                        
1195    Supply Chain Management Bachelor's in Supply C...
1196    Sociology Master's in Sociology 3 years experi...
1197    Chemistry Diploma in Chemistry 0 years experie...
1198    Biology Bachelor's in Biology 4 years experien...
1199    Sociology Master's in Sociology 0 years experi...
Name: resume_text, Length: 1200, dtype: object

In [38]:
X = df["resume_text"]
job_desc = df["Target_Job_Description"][0]
X[0]

"Cybersecurity Master's in Cybersecurity 0 years experience  Node.js, JavaScript, Deep Learning, Statistics, SQL Google Cloud Professional"

In [39]:
job_desc

'Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.'

In [40]:
documents = X.tolist() + [job_desc]
tfidf = TfidfVectorizer(stop_words = "english")
matrix = tfidf.fit_transform(documents)
scores = cosine_similarity(matrix[:-1], matrix[-1])
df["match_score"] = scores.flatten()*100


In [46]:
print(df["match_score"].describe())

count    808.000000
mean       3.732553
std        2.780013
min        0.749170
25%        1.328637
50%        2.835456
75%        5.393989
max       14.435911
Name: match_score, dtype: float64


In [42]:
df.sort_values(by = "match_score", ascending = False).head(10)

,Name,Age,Gender,Education_Level,Field_of_Study,Degrees,Institute_Name,Graduation_Year,Experience_Years,Current_Job_Title,Previous_Job_Titles,Skills,Certifications,Target_Job_Description,resume_text,match_score
1131,Diya Reddy,42,Male,Master's,Dentistry,Master's in Dentistry,Harvard University,2022,2,Customer Success Manager,NaN,"Organizational Skills, Risk Management, Psycho...",Certified Sales Professional,Seeking a challenging role as a Customer Succe...,Dentistry Master's in Dentistry 2 years experi...,14.435911
82,Vikram Taylor,25,Non-Binary,Master's,Software Engineering,Master's in Software Engineering,SRM Institute of Science and Technology,2020,4,Cybersecurity Analyst,Software Engineer,"Cloud Computing, React, AWS, DevOps, Docker, G...",Certified Information Systems Security Profess...,Targeting a Machine Learning Engineer position...,Software Engineering Master's in Software Engi...,11.685883
365,Aryan Williams,45,Female,Master's,Software Engineering,"Master's in Software Engineering, Bachelor's i...",All India Institute of Medical Sciences,2024,1,Software Developer,NaN,"GraphQL, SQL, Kafka, React, Blockchain, Azure,...",TensorFlow Developer Certificate,Looking for an opportunity as a Frontend Devel...,Software Engineering Master's in Software Engi...,11.584357
1070,Sophia Yang,40,Male,Bachelor's,Education,Bachelor's in Education,Stanford University,2019,4,Nurse,NaN,"Data Analysis, Customer Service, Compliance, A...","Google Analytics Certified, Licensed Professio...",Looking for a Business Development Manager rol...,Education Bachelor's in Education 4 years expe...,11.173095
258,Meera Park,25,Male,PhD,Software Engineering,PhD in Software Engineering,Vellore Institute of Technology,2015,4,Frontend Developer,AI Ethics Officer,"GraphQL, Git, Solidity, Statistics",Google Cloud Professional,Targeting a Frontend Developer position to uti...,Software Engineering PhD in Software Engineeri...,11.104366
141,Aditi Martinez,42,Non-Binary,Master's,Software Engineering,Master's in Software Engineering,University of Toronto,2017,4,Cloud Engineer,NaN,"Azure, Python, Kubernetes, Network Security","Google Data Analytics Professional, Oracle Cer...",Looking for a Cloud Engineer role where I can ...,Software Engineering Master's in Software Engi...,11.024261
352,Ashley Chen,33,Female,Bachelor's,Software Engineering,Bachelor's in Software Engineering,Amity University,2021,4,AI Engineer,Mobile Applications Developer,"AWS, Java, PyTorch, Machine Learning, Penetrat...",Project Management Professional,Seeking a challenging role as an AI Engineer t...,Software Engineering Bachelor's in Software En...,10.975589
739,Amanda Srinivasan,38,Female,Master's,Operations Management,Master's in Operations Management,Armed Forces Medical College,2022,1,Sales Representative,NaN,"Organizational Skills, Emotional Intelligence,...",NaN,Seeking a challenging role as a Human Resource...,Operations Management Master's in Operations M...,10.909711
852,Aryan Park,42,Non-Binary,Bachelor's,Pharmacy,Bachelor's in Pharmacy,Indian Institute of Technology Bombay,2019,2,Brand Manager,NaN,"Medical Knowledge, Organizational Skills, Heal...","SHRM Certified Professional, Registered Nurse ...",Looking for an opportunity as a Brand Manager ...,Pharmacy Bachelor's in Pharmacy 2 years experi...,10.808622
1119,Rahul Jackson,34,Non-Binary,PhD,Finance,PhD in Finance,Chennai University,2021,0,NaN,NaN,"Patient Care, Social Media Marketing, Budgetin...",NaN,Seeking a challenging role as a Content Manage...,Finance PhD in Finance 0 years experience Pat...,10.793793


In [43]:
df = df[df["match_score"] > 0]

In [44]:
def suitability(score):

    if score >= 8:
        return "Highly Suitable"

    elif score >= 4:
        return "Suitable"

    elif score >= 1.5:
        return "Moderate"

    else:
        return "Low Match"

df["Suitability"] = df["match_score"].apply(suitability)

In [47]:
df.sort_values(
    by="match_score",
    ascending=False
)[[
    "Name",
    "Current_Job_Title",
    "Skills",
    "match_score",
    "Suitability"
]].head(10)

,Name,Current_Job_Title,Skills,match_score,Suitability
1131,Diya Reddy,Customer Success Manager,"Organizational Skills, Risk Management, Psycho...",14.435911,Highly Suitable
82,Vikram Taylor,Cybersecurity Analyst,"Cloud Computing, React, AWS, DevOps, Docker, G...",11.685883,Highly Suitable
365,Aryan Williams,Software Developer,"GraphQL, SQL, Kafka, React, Blockchain, Azure,...",11.584357,Highly Suitable
1070,Sophia Yang,Nurse,"Data Analysis, Customer Service, Compliance, A...",11.173095,Highly Suitable
258,Meera Park,Frontend Developer,"GraphQL, Git, Solidity, Statistics",11.104366,Highly Suitable
141,Aditi Martinez,Cloud Engineer,"Azure, Python, Kubernetes, Network Security",11.024261,Highly Suitable
352,Ashley Chen,AI Engineer,"AWS, Java, PyTorch, Machine Learning, Penetrat...",10.975589,Highly Suitable
739,Amanda Srinivasan,Sales Representative,"Organizational Skills, Emotional Intelligence,...",10.909711,Highly Suitable
852,Aryan Park,Brand Manager,"Medical Knowledge, Organizational Skills, Heal...",10.808622,Highly Suitable
1119,Rahul Jackson,NaN,"Patient Care, Social Media Marketing, Budgetin...",10.793793,Highly Suitable
